# 🍄 Boscdex Classifier Training on Google Colab (Free T4 GPU)

This notebook fine-tunes an on-device image classifier for **Boscdex** using **MobileNetV2** and **Transfer Learning**.

### 🚀 Highlights:
1. **1-Click Google Drive Ingestion:** Automatically finds and unzips `boscdex_dataset.zip` from your Google Drive into `./dataset/`.
2. **Hybrid / Dual Sourcing:** Uses your field photos from `test/assets/`, with optional iNaturalist expansion for underrepresented species.
3. **Decoupled Architecture:** Exports both the **TFLite feature extractor** (`mushroom_classifier.tflite`) and the **binary classification head** (`mushroom_classifier_head.bin`), guaranteed 100% compatible with the Boscdex Flutter app!

### ⚡ Quick Start:
1. Ensure runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Click **Runtime > Run all** (or run each cell sequentially).
3. When finished, your browser will automatically download the 3 updated model files!

In [9]:
# Step 1: Verify Hardware & GPU Allocation
import sys
import platform
import shutil
import tensorflow as tf

print(f"Python Version:      {sys.version.split()[0]}")
print(f"Platform:            {platform.platform()}")
print(f"TensorFlow Version:  {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"✅ GPU Acceleration Active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("⚠️ No GPU detected. For 10x faster training, select: Runtime > Change runtime type > T4 GPU.")


Python Version:      3.13.15
Platform:            Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow Version:  2.21.0
✅ GPU Acceleration Active: /physical_device:GPU:0
Tesla T4, 15360 MiB, 13692 MiB


In [10]:
# Step 2: Install Required Libraries
!pip install -q pyinaturalist requests pillow tqdm


In [11]:
# Step 3: Species Catalog & Directory Structure
import os
from pathlib import Path

DATASET_DIR = Path("./dataset")
EXPORT_DIR = Path("./export")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 40 Catalan Fungi Taxa in Boscdex (folder_id, inaturalist_taxon_id, catalan_name)
SPECIES_CATALOG = [
    ("lactarius_deliciosus", 54595, "Rovelló / Pinetell"),
    ("boletus_edulis", 48701, "Cep"),
    ("cantharellus_cibarius", 47347, "Rossinyol"),
    ("amanita_caesarea", 204588, "Ou de reig"),
    ("craterellus_cornucopioides", 48607, "Trompeta de la mort"),
    ("macrolepiota_procera", 63401, "Apagallums"),
    ("tricholoma_terreum", 63438, "Fredolic"),
    ("calocybe_gambosa", 122135, "Moixernó"),
    ("pleurotus_ostreatus", 48494, "Gírgola"),
    ("hydnum_repandum", 48682, "Llengua de bou"),
    ("hygrophorus_latitabundus", 353142, "Llenega negra"),
    ("craterellus_lutescens", 67707, "Camagroc"),
    ("suillus_luteus", 53488, "Llenegot"),
    ("russula_cyanoxantha", 63584, "Llora verda"),
    ("amanita_muscaria", 48715, "Reig bord"),
    ("omphalotus_olearius", 217916, "Gírgola d'olivera"),
    ("entoloma_sinuatum", 921907, "Pèrfid"),
    ("amanita_phalloides", 52135, "Farinera borda"),
    ("amanita_verna", 58694, "Cicuta blanca"),
    ("cortinarius_orellanus", 202404, "Cortinari de muntanya"),
    ("lactarius_sanguifluus", 118207, "Rovelló de sang / Pinetell de sang"),
    ("chroogomphus_rutilus", 194275, "Cama de perdiu"),
    ("marasmius_oreades", 47285, "Carrereta / Cama-sec"),
    ("lactarius_chrysorrheus", 154044, "Lleterola de llet groga / Fals rovelló"),
    ("lepista_nuda", 351427, "Pé blau / Pua blava"),
    ("rubroboletus_satanas", 544548, "Mataparent de Satanàs"),
    ("russula_virescens", 118228, "Palometa / Llora pigada"),
    ("hygrophorus_eburneus", 67331, "Llenega blanca / Mocosa blanca"),
    ("tricholoma_pardinum", 341772, "Fredolic bord / Griset bord"),
    ("morchella_esculenta", 48089, "Múrgola / Rabassola"),
    ("agaricus_campestris", 143550, "Xampinyó silvestre / Guisana"),
    ("agaricus_xanthodermus", 118394, "Xampinyó groguenc / Xampinyó pudent"),
    ("psathyrella_candolleana", 1179922, "Candol·lea / Psatirel·la de Candolle"),
    ("suillus_granulatus", 179067, "Molleric clapat / Pinetell de pi blanc"),
    ("hygrophorus_russula", 350225, "Carlet / Escarlet"),
    ("boletus_aereus", 343274, "Cep negre / Sureny fosc"),
    ("coprinus_comatus", 47392, "Bolet de tinta / Braguer de vaca"),
    ("lactarius_torminosus", 68310, "Cabra / Rovelló bord"),
    ("amanita_pantherina", 48418, "Pantera / Reig bord pigat"),
    ("gyromitra_esculenta", 85120, "Bolet de greix / Falsa múrgola"),
]

for s_id, _, _ in SPECIES_CATALOG:
    (DATASET_DIR / s_id).mkdir(parents=True, exist_ok=True)

print(f"✓ Configured {len(SPECIES_CATALOG)} species directories in {DATASET_DIR.resolve()}")


✓ Configured 40 species directories in /content/dataset


In [12]:
# Step 4: Dataset Ingestion (Google Drive boscdex_dataset.zip or local upload)
import zipfile
import shutil

# 1. Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"ℹ️ Google Drive mount note: {e}")

# 2. Check candidate locations for boscdex_dataset.zip
candidate_paths = [
    Path("/content/drive/MyDrive/boscdex_dataset.zip"),
    Path("/content/drive/MyDrive/boscdex/boscdex_dataset.zip"),
    Path("./boscdex_dataset.zip"),
]

found_archive = next((p for p in candidate_paths if p.exists()), None)

if found_archive:
    print(f"📦 Found dataset archive: {found_archive} ({found_archive.stat().st_size / (1024*1024):.1f} MB)")
    print("Extracting images into ./dataset/...")
    with zipfile.ZipFile(found_archive, "r") as zf:
        zf.extractall(DATASET_DIR)
    print("✅ Extraction complete!")
else:
    print("⚠️ No boscdex_dataset.zip found in Google Drive root or current working directory.")
    print("   Upload boscdex_dataset.zip to Google Drive (MyDrive/) or click the folder icon on the left to upload it.")

# 3. Print census
print()
print("📊 Dataset Census by Species:")
total_images = 0
for s_id, _, name in SPECIES_CATALOG:
    folder = DATASET_DIR / s_id
    count = len(list(folder.glob("*.*"))) if folder.exists() else 0
    total_images += count
    print(f"  • {s_id.ljust(28)}: {count:3d} images ({name})")
print(f"Total images available: {total_images}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully.
📦 Found dataset archive: /content/drive/MyDrive/boscdex/boscdex_dataset.zip (155.3 MB)
Extracting images into ./dataset/...
✅ Extraction complete!

📊 Dataset Census by Species:
  • lactarius_deliciosus        :  40 images (Rovelló / Pinetell)
  • boletus_edulis              :  40 images (Cep)
  • cantharellus_cibarius       :  40 images (Rossinyol)
  • amanita_caesarea            :  47 images (Ou de reig)
  • craterellus_cornucopioides  :  40 images (Trompeta de la mort)
  • macrolepiota_procera        :  40 images (Apagallums)
  • tricholoma_terreum          :  11 images (Fredolic)
  • calocybe_gambosa            :   5 images (Moixernó)
  • pleurotus_ostreatus         :  40 images (Gírgola)
  • hydnum_repandum             :  40 images (Llengua de bou)
  • hygrophorus_latitabundus    :  22 images (Llenega negra)
  • crater

In [13]:
# Step 5: iNaturalist Top-Up for Balanced Classes (Option B: 40+ photos per species)
import time
import io
from pathlib import Path
from PIL import Image
import requests
from tqdm.notebook import tqdm
from pyinaturalist import get_observations

# Target minimum images per species:
# 40 images per species x 40 classes = ~1,600 balanced training photos
MIN_IMAGES_PER_SPECIES = 40

PLACE_CATALONIA = 7161
PLACE_SPAIN = 6774
PLACE_EUROPE = 97391

HEADERS = {"User-Agent": "Boscdex-Fungi-Classifier/1.0"}

def download_and_save(url, dest_path):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        if resp.status_code == 200:
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            min_dim = min(w, h)
            left = (w - min_dim) // 2
            top = (h - min_dim) // 2
            img = img.crop((left, top, left + min_dim, top + min_dim))
            img = img.resize((224, 224), Image.Resampling.LANCZOS)
            img.save(dest_path, "JPEG", quality=90)
            return True
    except Exception:
        pass
    return False

def count_valid_images(folder):
    if not folder.exists():
        return 0
    return len([p for p in folder.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

def top_up_species(s_id, taxon_id, target_count):
    folder = DATASET_DIR / s_id
    folder.mkdir(parents=True, exist_ok=True)
    existing = count_valid_images(folder)
    if existing >= target_count:
        return existing

    downloaded = existing
    # Progressive geographical widening: Catalonia -> Spain -> Europe -> Worldwide
    for pid in [PLACE_CATALONIA, PLACE_SPAIN, PLACE_EUROPE, None]:
        if downloaded >= target_count:
            break
        for page in range(1, 8):
            if downloaded >= target_count:
                break
            try:
                kwargs = {
                    "taxon_id": taxon_id,
                    "quality_grade": "research",
                    "photos": True,
                    "per_page": 50,
                    "page": page,
                }
                if pid is not None:
                    kwargs["place_id"] = pid

                obs = get_observations(**kwargs)
                results = obs.get("results", [])
                if not results:
                    break
                urls = []
                for o in results:
                    for p in o.get("photos", []):
                        u = p.get("url", "").replace("square", "medium")
                        if u:
                            urls.append(u)

                for u in urls:
                    if downloaded >= target_count:
                        break
                    downloaded += 1
                    dest = folder / f"inat_{downloaded:04d}.jpg"
                    if not download_and_save(u, dest):
                        downloaded -= 1
                    else:
                        time.sleep(0.05)
            except Exception as e:
                break
    return downloaded

if MIN_IMAGES_PER_SPECIES > 0:
    print(f"⚖️ Balancing dataset: Ensuring all {len(SPECIES_CATALOG)} species have at least {MIN_IMAGES_PER_SPECIES} images...")
    for s_id, t_id, name in tqdm(SPECIES_CATALOG, desc="Balancing Taxa"):
        count = top_up_species(s_id, t_id, target_count=MIN_IMAGES_PER_SPECIES)
        print(f"  • {s_id.ljust(28)}: {count:2d} images ({name})")
        time.sleep(0.1)

    total = sum(count_valid_images(DATASET_DIR / s_id) for s_id, _, _ in SPECIES_CATALOG)
    print(f"✅ Dataset fully balanced! Total images available: {total}")
else:
    print("Skipping iNaturalist download (MIN_IMAGES_PER_SPECIES=0).")


⚖️ Balancing dataset: Ensuring all 40 species have at least 40 images...


Balancing Taxa:   0%|          | 0/40 [00:00<?, ?it/s]

  • lactarius_deliciosus        : 40 images (Rovelló / Pinetell)
  • boletus_edulis              : 40 images (Cep)
  • cantharellus_cibarius       : 40 images (Rossinyol)
  • amanita_caesarea            : 47 images (Ou de reig)
  • craterellus_cornucopioides  : 40 images (Trompeta de la mort)
  • macrolepiota_procera        : 40 images (Apagallums)
  • tricholoma_terreum          : 11 images (Fredolic)
  • calocybe_gambosa            :  5 images (Moixernó)
  • pleurotus_ostreatus         : 40 images (Gírgola)
  • hydnum_repandum             : 40 images (Llengua de bou)


  • hygrophorus_latitabundus    : 22 images (Llenega negra)
  • craterellus_lutescens       : 40 images (Camagroc)
  • suillus_luteus              : 40 images (Llenegot)
  • russula_cyanoxantha         : 10 images (Llora verda)
  • amanita_muscaria            : 40 images (Reig bord)
  • omphalotus_olearius         : 11 images (Gírgola d'olivera)
  • entoloma_sinuatum           : 40 images (Pèrfid)
  • amanita_phalloides          : 40 images (Farinera borda)
  • amanita_verna               : 40 images (Cicuta blanca)
  • cortinarius_orellanus       : 40 images (Cortinari de muntanya)
  • lactarius_sanguifluus       : 40 images (Rovelló de sang / Pinetell de sang)
  • chroogomphus_rutilus        : 40 images (Cama de perdiu)
  • marasmius_oreades           : 40 images (Carrereta / Cama-sec)
  • lactarius_chrysorrheus      : 40 images (Lleterola de llet groga / Fals rovelló)
  • lepista_nuda                : 40 images (Pé blau / Pua blava)
  • rubroboletus_satanas        : 40 images (Matap

In [14]:
# Step 6: Train & Fine-Tune MobileNetV2 (Native Keras Transfer Learning)
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Acceleration:    {'✅ Available (' + gpus[0].name + ')' if gpus else '⚠️ CPU Only'}")

# Clean up empty directories so only populated species folders are loaded
for p in list(DATASET_DIR.iterdir()):
    if p.is_dir() and not any(p.glob("*.*")):
        p.rmdir()

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Loading dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
print(f"Classes ({len(class_names)}): {class_names}")

# Robust data augmentation pipeline (includes brightness & contrast for screen scans and variable lighting)
AUTOTUNE = tf.data.AUTOTUNE
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.20),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomBrightness(0.20),
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# Base MobileNetV2 pre-trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

# Architecture with Named Layers for Decoupled Extraction
# Note: mobilenet_v2.preprocess_input scales [0, 255] float input to [-1, 1] internally!
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
gap = tf.keras.layers.GlobalAveragePooling2D(name="gap_features")(x)
dropout = tf.keras.layers.Dropout(0.25, name="dropout")(gap)
head = tf.keras.layers.Dense(len(class_names), name="classifier_head")(dropout)
outputs = tf.keras.layers.Softmax(name="softmax")(head)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="boscdex_classifier")

# Callbacks for optimal convergence
reduce_lr_p1 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print()
print("🚀 Phase 1: Training top classification head (15 epochs)...")
model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[reduce_lr_p1])

print()
print("🚀 Phase 2: Fine-tuning top MobileNetV2 layers (up to 20 epochs with early stopping)...")
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

reduce_lr_p2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[reduce_lr_p2, early_stop],
)

val_loss, val_acc = model.evaluate(val_ds)
print()
print(f"📊 Final Model Validation Accuracy: {val_acc * 100:.2f}%")


TensorFlow Version: 2.21.0
GPU Acceleration:    ✅ Available (/physical_device:GPU:0)
Loading dataset...
Found 1471 files belonging to 47 classes.
Using 1177 files for training.
Found 1471 files belonging to 47 classes.
Using 294 files for validation.
Classes (47): ['agaricus_campestris', 'agaricus_xanthodermus', 'amanita_caesarea', 'amanita_muscaria', 'amanita_pantherina', 'amanita_phalloides', 'amanita_verna', 'amanita_virosa', 'boletus_aereus', 'boletus_edulis', 'calocybe_gambosa', 'cantharellus_cibarius', 'chroogomphus_rutilus', 'coprinus_comatus', 'cortinarius_orellanus', 'craterellus_cornucopioides', 'craterellus_lutescens', 'entoloma_sinuatum', 'galerina_marginata', 'gyromitra_esculenta', 'hydnum_repandum', 'hygrophorus_eburneus', 'hygrophorus_latitabundus', 'hygrophorus_personii', 'hygrophorus_russula', 'lactarius_chrysorrheus', 'lactarius_deliciosus', 'lactarius_sanguifluus', 'lactarius_semisanguifluus', 'lactarius_torminosus', 'lepista_nuda', 'macrolepiota_procera', 'marasmius

In [15]:
# Step 7: Export Decoupled Model Assets for Boscdex Flutter App
import struct
import json

EXPORT_DIR = Path("./export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export labels.txt
labels_path = EXPORT_DIR / "mushroom_labels.txt"
with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
print(f"✓ Exported {len(class_names)} classes to: {labels_path}")

# 2. Export GAP Feature Extractor (Input: [1, 224, 224, 3], Output: [1, 1280] Float32)
# This matches TfliteClassifierService in Flutter exactly!
gap_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("gap_features").output)
converter = tf.lite.TFLiteConverter.from_keras_model(gap_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_gap = converter.convert()

tflite_path = EXPORT_DIR / "mushroom_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_gap)
print(f"✓ Exported GAP feature extractor: {tflite_path} ({tflite_path.stat().st_size / (1024*1024):.2f} MB)")

# 3. Export Portable Binary Classification Head (classifier_head.bin)
# Format: N float32 biases followed by N x 1280 float32 weights (little-endian)
dense_layer = model.get_layer("classifier_head")
weights, biases = dense_layer.get_weights()  # weights shape: (1280, N), biases shape: (N,)

head_bin_path = EXPORT_DIR / "mushroom_classifier_head.bin"
with open(head_bin_path, "wb") as f:
    # Write biases (len(class_names) floats)
    for b in biases:
        f.write(struct.pack("<f", float(b)))
    # Write weights (len(class_names) classes x 1280 features)
    for c in range(len(class_names)):
        for feature_idx in range(1280):
            w = weights[feature_idx, c]
            f.write(struct.pack("<f", float(w)))

print(f"✓ Exported binary classification head: {head_bin_path} ({head_bin_path.stat().st_size / 1024:.1f} KB)")

# 4. Also export full end-to-end model for reference
full_converter = tf.lite.TFLiteConverter.from_keras_model(model)
full_tflite = full_converter.convert()
full_path = EXPORT_DIR / "mushroom_classifier_full.tflite"
with open(full_path, "wb") as f:
    f.write(full_tflite)
print(f"✓ Exported full reference model: {full_path} ({full_path.stat().st_size / (1024*1024):.2f} MB)")


✓ Exported 47 classes to: export/mushroom_labels.txt
Saved artifact at '/tmp/tmpjkrlprl9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 1280), dtype=tf.float32, name=None)
Captures:
  138199029111440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029113744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029112208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029113552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029113360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029112976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029117392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029113168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029108752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138199029113936: TensorSpec(s

In [16]:
# Step 8: Export & Download Model Files
import shutil

# 1. If running locally or in VS Code with workspace root accessible, auto-copy to assets/models/
local_assets_targets = [Path("assets/models"), Path("../assets/models")]
for target in local_assets_targets:
    if target.exists() and target.is_dir():
        for f in [tflite_path, head_bin_path, labels_path]:
            shutil.copy2(f, target / f.name)
        print(f"✅ Automatically copied trained models to local workspace: {target.resolve()}/")
        break

# 2. If Google Drive is mounted, also back up to Google Drive
drive_models_dir = Path("/content/drive/MyDrive/boscdex_mushroom_models")
if Path("/content/drive/MyDrive").exists():
    drive_models_dir.mkdir(parents=True, exist_ok=True)
    for f in [tflite_path, head_bin_path, labels_path]:
        shutil.copy2(f, drive_models_dir / f.name)
    print(f"💾 Saved a permanent copy to Google Drive: {drive_models_dir}/")

# 3. Attempt direct browser download (works in web Colab)
try:
    from google.colab import files
    print("Triggering browser download of Boscdex model assets...")
    files.download(str(tflite_path))
    files.download(str(head_bin_path))
    files.download(str(labels_path))
except Exception as e:
    print(f"Browser download skipped/not supported: {e}")

print()
print("🎉 Done! Copy these 3 files into your local project:")
print("   1. assets/models/mushroom_classifier.tflite")
print("   2. assets/models/mushroom_classifier_head.bin")
print("   3. assets/models/mushroom_labels.txt")
print()
print("Then verify with:")
print("   flutter test test/unit_tests/domain/mushroom_model_real_images_test.dart")

💾 Saved a permanent copy to Google Drive: /content/drive/MyDrive/boscdex_mushroom_models/
Triggering browser download of Boscdex model assets...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Done! Copy these 3 files into your local project:
   1. assets/models/mushroom_classifier.tflite
   2. assets/models/mushroom_classifier_head.bin
   3. assets/models/mushroom_labels.txt

Then verify with:
   flutter test test/unit_tests/domain/mushroom_model_real_images_test.dart
